# GLADIUS applied workflow

This notebook fits the public, paper-reference GLADIUS path on an anchored dynamic-choice panel. It checks the pre-estimation diagnostics, reports reward and policy outputs, re-solves a reward counterfactual with supplied planning transitions, and verifies serialization. The transition tensor is stored for planning; GLADIUS does not use it to estimate the reward.

In [ ]:
import pickle
import warnings
from pathlib import Path

import jax.numpy as jnp
import numpy as np
import pandas as pd

import econirl
from econirl import GLADIUS
from econirl.core.reward_spec import RewardSpec

checkout_root = Path.cwd().resolve().parents[1]
module_outside_checkout = not Path(econirl.__file__).resolve().is_relative_to(checkout_root)
print(f'Installed package import: {module_outside_checkout}')
print(f'Package version: {econirl.__version__}')
print(f'Package module: {econirl.__file__}')

## Build an anchored panel

Action 0 is the anchor and has known reward zero in every state. Action 1 moves the state forward. The feature basis describes the action-1 reward contrast.

In [ ]:
n_states, n_actions = 6, 2
rows = []
for individual in range(24):
    state = individual % n_states
    for period in range(12):
        action = (individual + period) % n_actions
        next_state = state if action == 0 else (state + 1) % n_states
        rows.append({
            'id': individual, 'period': period, 'state': state,
            'action': action, 'next_state': next_state,
        })
        state = next_state
panel = pd.DataFrame(rows)

feature_matrix = np.zeros((n_states, n_actions, 1), dtype=np.float32)
feature_matrix[:, 1, 0] = 1.0
features = RewardSpec(jnp.asarray(feature_matrix), names=['action_one'])

transitions = np.zeros((n_actions, n_states, n_states))
for state in range(n_states):
    transitions[0, state, state] = 1.0
    transitions[1, state, (state + 1) % n_states] = 1.0

panel.head()

In [ ]:
model = GLADIUS(
    n_actions=n_actions, discount=0.90,
    q_hidden_dim=16, q_num_layers=1,
    ev_hidden_dim=16, ev_num_layers=1,
    batch_size=64, max_epochs=30, patience=10,
    anchor_action=0, anchor_rewards=tuple([0.0] * n_states),
    seed=7,
)
with warnings.catch_warnings():
    warnings.simplefilter('ignore', RuntimeWarning)
    warnings.simplefilter('ignore', UserWarning)
    model.fit(
        panel, state='state', action='action', id='id',
        features=features, transitions=transitions,
    )

assert model.objective_ == 'paper_minimax'
assert np.isfinite(model.reward_).all()
assert np.isfinite(model.policy_).all()
model.diagnostics_

In [ ]:
print(model.summary())
print('anchor mean reward:', float(model.reward_[:, 0].mean()))
print('policy row sums:', model.policy_.sum(axis=1))

## Counterfactual and serialization

A reward intervention is structurally supported only because the fit supplied both the known anchor reward and a baseline transition tensor for planning.

In [ ]:
reward_delta = np.zeros_like(model.reward_)
reward_delta[:, 1] = 0.5
counterfactual = model.counterfactual(
    reward_delta=reward_delta,
    description='action-1 subsidy',
)
print('maximum policy change:', float(np.max(np.abs(counterfactual.policy_change))))
print('mean welfare change:', counterfactual.welfare_change)
assert np.max(np.abs(counterfactual.policy_change)) > 0

restored = pickle.loads(pickle.dumps(model))
np.testing.assert_allclose(restored.predict_proba(np.arange(n_states)), model.policy_)
np.testing.assert_allclose(restored.reward_, model.reward_)
print('serialization: exact supported-output parity')

## Interpretation boundary

The `se_` and `pvalues_` fields attached to a feature projection are descriptive regression diagnostics, not sampling uncertainty. Use `compute_se=True` for whole-trajectory bootstrap intervals. The Table 2 replication separately labels the authors' simulation-only best-true-MAPE epoch selection; deployable fits never use that oracle.